In [1]:
import re
from collections import defaultdict
from functools import cache
from typing import Annotated, Literal, Self

import pandas as pd
import pulp
from pydantic import BaseModel, BeforeValidator, TypeAdapter

In [2]:
bracket_raw = pd.read_csv("00_bracket.csv")
lookup_raw = pd.read_csv("00_lookup.csv")
ratings_raw = pd.read_csv("01_ratings.csv")
tips_raw = pd.read_csv("02_tips.csv")

In [3]:
tips = tips_raw.groupby(["player", "stage"])["team"].apply(list).unstack().fillna("").apply(list)

In [4]:
ratings = (
    lookup_raw.merge(ratings_raw, how="left", left_on="eloratings_name", right_on="team")
    .set_index("fifa_code")["rating"]
    .to_dict()
)

In [5]:
def parse_team(s):
    if re.match(R"[WL]\d+", s):
        return PrevMatch.from_string(s)
    return s


class PrevMatch(BaseModel):
    match_number: int
    team: Literal["W", "L"]

    @classmethod
    def from_string(cls, s: str):
        return PrevMatch(match_number=int(s[1:]), team=s[0])


class BracketMatch(BaseModel):
    match_number: int
    home: Annotated[str | PrevMatch, BeforeValidator(parse_team)]
    away: Annotated[str | PrevMatch, BeforeValidator(parse_team)]
    stage: str

In [6]:
matches = {
    match.match_number: match
    for match in TypeAdapter(list[BracketMatch]).validate_python(bracket_raw.to_dict(orient="records"))
}

In [7]:
class MatchTeam(BaseModel):
    team: str
    source: PrevMatch | None = None

    def replace_source(self, source: PrevMatch) -> Self:
        return self.model_copy(update={"source": source})


@cache
def possible_teams_for_match(match_number: int) -> list[MatchTeam]:
    match = matches[match_number]
    out = []
    if not isinstance(match.home, str):
        out.extend(mt.replace_source(match.home) for mt in possible_teams_for_match(match.home.match_number))
    else:
        out.append(MatchTeam(team=match.home))

    if not isinstance(match.away, str):
        out.extend(mt.replace_source(match.away) for mt in possible_teams_for_match(match.away.match_number))
    else:
        out.append(MatchTeam(team=match.away))

    return out


def var_name(match, team, kind):
    return f"m{match:03d}_{team}_{kind}"

In [8]:
prob = pulp.LpProblem("Bracket", pulp.LpMinimize)

winner_vars = defaultdict(dict)
participant_vars = defaultdict(dict)
participants_by_stage = {match.stage: defaultdict(list) for match in matches.values()}
winners_by_stage = {match.stage: defaultdict(list) for match in matches.values()}

ratings_difference = 0

for match in sorted(matches):
    match_winners = []
    match_participants = []
    for match_team in possible_teams_for_match(match):
        team = match_team.team
        source = match_team.source
        rating = ratings[team]
        winner_var = prob.add_variable(var_name(match, team, "winner"), 0, 1, pulp.LpInteger)
        participant_var = prob.add_variable(var_name(match, team, "participant"), 0, 1, pulp.LpInteger)

        ratings_difference += (participant_var - 2 * winner_var) * rating

        prob += participant_var >= winner_var
        if source:
            source_part_var = participant_vars[source.match_number][team]
            prob += source_part_var >= participant_var
            source_winner_var = winner_vars[source.match_number][team]
            if source.team == "W":
                prob += source_winner_var == participant_var
            else:
                prob += (1 - source_winner_var) >= participant_var
        else:
            prob += participant_var == 1

        winner_vars[match][team] = winner_var
        participant_vars[match][team] = participant_var

        stage = matches[match].stage
        winners_by_stage[stage][team].append(winner_var)
        participants_by_stage[stage][team].append(participant_var)

        match_winners.append(winner_var)
        match_participants.append(participant_var)
    prob += pulp.lpSum(match_winners) == 1
    prob += pulp.lpSum(match_participants) == 2

prob += ratings_difference

winner_vars = dict(winner_vars)
participant_vars = dict(participant_vars)

In [9]:
PARTICIPANT_SCORING = {"R32": 1, "R16": 2, "quarter": 4, "semi": 6, "final": 8}
WINNER_SCORING = {"third": ("third", 4), "final": ("winner", 12)}

In [10]:
players = sorted(tips_raw["player"].unique())

player_vars = prob.add_variable_dict("player_score", players, cat=pulp.LpInteger)
for player in players:
    temp_var = 0
    for match in matches.values():
        stage = match.stage
        if stage in PARTICIPANT_SCORING:
            participant_points = PARTICIPANT_SCORING[stage]
            participant_teams = tips.loc[player, stage]
            temp_var += participant_points * pulp.lpSum(
                [
                    prob.variablesDict().get(var_name(match.match_number, team, "participant"), 0)
                    for team in participant_teams
                ]
            )
        if stage in WINNER_SCORING:
            winner_stage, winner_pts = WINNER_SCORING[stage]
            winner_teams = tips.loc[player, winner_stage]
            temp_var += winner_pts * pulp.lpSum(
                [prob.variablesDict().get(var_name(match.match_number, team, "winner"), 0) for team in winner_teams]
            )
    prob += player_vars[player] == temp_var

In [11]:
prob.solve()

winners = {}
participants = defaultdict(list)

for var in prob.variables():
    if var.name == "__dummy":
        continue
    if var.value() != 1:
        continue
    match_raw, team, kind = var.name.split("_")
    match = int(match_raw[1:])
    if kind == "winner":
        winners[match] = team
    elif kind == "participant":
        participants[match].append(team)
    else:
        raise ValueError(var.name)

participants = dict(participants)

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /home/adam/repos/wc26-tipping/.venv/lib64/python3.14/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/9b8b2308b7184d879198aeca4d609294-pulp.mps -timeMode elapsed -solve -printingOptions all -solution /tmp/9b8b2308b7184d879198aeca4d609294-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 638 COLUMNS
At line 4692 RHS
At line 5326 BOUNDS
At line 5736 ENDATA
Problem MODEL has 633 rows, 409 columns and 2851 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is -5267 - 0.00 seconds
Cgl0003I 0 fixed, 0 tightened bounds, 96 strengthened rows, 0 substitutions
Cgl0003I 0 fixed, 0 tightened bounds, 29 strengthened rows, 0 substitutions
Cgl0003I 0 fixed, 0 tightened bounds, 29 strengthened rows, 0 substitutions
Cgl0003I 0 fixed, 0 tightened bounds, 29 strengthened rows, 0 substitutions
Cgl0003I 0 fi

In [12]:
for match in sorted(matches):
    winner = winners[match]
    home, away = participants[match]
    print(
        f"{match:3} {matches[match].stage:7} {'👑' if winner == home else '  '}{home} ({ratings[home]}) vs {'👑' if winner == away else '  '}{away} ({ratings[away]})"
    )

 73 R32     👑CAN (1764) vs   RSA (1559)
 74 R32     👑GER (1916) vs   PAR (1815)
 75 R32       MAR (1877) vs 👑NED (1980)
 76 R32     👑BRA (2009) vs   JPN (1910)
 77 R32     👑FRA (2123) vs   SWE (1742)
 78 R32       CIV (1743) vs 👑NOR (1918)
 79 R32       ECU (1902) vs 👑MEX (1912)
 80 R32       COD (1712) vs 👑ENG (2038)
 81 R32       BIH (1622) vs 👑USA (1781)
 82 R32     👑BEL (1884) vs   SEN (1842)
 83 R32       CRO (1905) vs 👑POR (1990)
 84 R32       AUT (1836) vs 👑ESP (2144)
 85 R32       ALG (1785) vs 👑SUI (1914)
 86 R32     👑ARG (2148) vs   CPV (1622)
 87 R32     👑COL (2004) vs   GHA (1575)
 88 R32     👑AUS (1800) vs   EGY (1742)
 89 R16     👑FRA (2123) vs   GER (1916)
 90 R16       CAN (1764) vs 👑NED (1980)
 91 R16     👑BRA (2009) vs   NOR (1918)
 92 R16     👑ENG (2038) vs   MEX (1912)
 93 R16     👑ESP (2144) vs   POR (1990)
 94 R16     👑BEL (1884) vs   USA (1781)
 95 R16     👑ARG (2148) vs   AUS (1800)
 96 R16     👑COL (2004) vs   SUI (1914)
 97 quarter 👑FRA (2123) vs   NED (1980)


In [13]:
print(pd.Series({name: var.value() for name, var in player_vars.items()}).sort_values(ascending=False).to_string())

Luce                           122.0
Claude (AI)                    119.0
Jon                            112.0
Martin L                       111.0
Dave L                         106.0
Dan                            102.0
Smith                          101.0
Adam B                         101.0
Doug                            99.0
Giuseppe                        99.0
Kuba                            99.0
Albert                          98.0
Antoni Brdarovski               97.0
ESPN (Ryan O'Hanlon's tips)     94.0
Sherram                         91.0
Tamara                          90.0
Tom (9a0)                       88.0
Mikey                           88.0
Haydos                          88.0
Estevao                         88.0
Damian                          83.0
MarcinC                         83.0
Downsy                          83.0
Tom (c66)                       82.0
BlueLaGoons                     68.0
